# Machine Learning 3 - Support Vector Machines

A SVM classifier builds a set of hyper-planes to try and separate the data by maximizing the distance between the borders and the data points.

![SVM](http://scikit-learn.org/stable/_images/sphx_glr_plot_separating_hyperplane_0011.png "Decision border in an SVM")

This separation is generally not possible to achieve in the original data space. Therefore, the first step of the SVM is to project the data into a high or infinite dimensions space in which this linear separation can be done. The projection can be done with linear, polynomial, or more comonly "RBF" kernels.

In [ ]:
from lab_tools import CIFAR10, evaluate_classifier, get_hog_image

dataset = CIFAR10('./CIFAR10')

**Build a simple SVM** using [the SVC (Support Vector Classfiication) from sklearn](http://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC). 
**Train** it on the CIFAR dataset.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

X_train = dataset.train['hog']
y_train = dataset.train['labels']
X_test = dataset.test['hog']
y_test = dataset.test['labels']

# Simple SVM with default parameters (RBF kernel)
clf = SVC()
clf.fit(X_train, y_train)

# Evaluate on the test set
y_pred = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

**Explore the classifier**. How many support vectors are there? What are support vectors?

In [ ]:
# Total number of support vectors
all_support_vectors = clf.support_vectors_
print("Total number of support vectors:", all_support_vectors.shape[0])
print("Shape of support vectors:", all_support_vectors.shape)

# Number of support vectors per class
vectors_per_class = clf.n_support_
print("Support vectors per class:", vectors_per_class)
print("Classes:", clf.classes_)

# Support vectors are training samples that lie on the margin
# or inside it (and the misclassified ones).
# They are the only training points that define the decision boundary:
# if we removed all the other training samples, we would obtain the same SVM.

**Try to find the best "C" (error penalty) and "gamma" parameters** using cross-validation. What influence does "C" have on the number of support vectors?

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

# SVM cross-validation is expensive, so we work on a subset of the training set
rng = np.random.RandomState(42)
n_subset = 3000
idx = rng.choice(len(X_train), n_subset, replace=False)
X_sub = X_train[idx]
y_sub = y_train[idx]

C_values = [0.1, 1, 10, 100]
gamma_values = ['scale', 0.01, 0.1]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_score = 0
best_params = None

print("Cross-validation results:")
for C in C_values:
    for gamma in gamma_values:
        clf_cv = SVC(C=C, gamma=gamma)
        scores = cross_val_score(clf_cv, X_sub, y_sub, cv=cv, scoring='accuracy')
        mean_score = scores.mean()
        print(f"C = {C}, gamma = {gamma} -> mean CV accuracy = {mean_score:.4f}")
        if mean_score > best_score:
            best_score = mean_score
            best_params = (C, gamma)

print("\nBest parameters: C =", best_params[0], ", gamma =", best_params[1])
print("Best CV accuracy:", best_score)

# Influence of C on the number of support vectors
print("\nInfluence of C on the number of support vectors:")
for C in [0.01, 0.1, 1, 10, 100]:
    clf_c = SVC(C=C, gamma='scale')
    clf_c.fit(X_sub, y_sub)
    print(f"C = {C} -> total support vectors = {clf_c.support_vectors_.shape[0]}")

# A small C makes a "softer" margin: more samples are inside the margin,
# so more support vectors. A large C makes a "harder" margin: fewer samples
# violate it, so fewer support vectors (but more risk of overfitting).

# Comparing algorithms

Using the best hyper-parameters that you found for each of the algorithms (kNN, Decision Trees, Random Forests, MLP, SVM):

* Re-train the models on the full training set.
* Compare their results on the test set.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Best hyper-parameters found in the previous labs
models = {
    "kNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree (max_depth=10)": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest (n=100)": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP (100, 100)": MLPClassifier(hidden_layer_sizes=(100, 100), max_iter=300, random_state=42),
    "SVM (best from CV)": SVC(C=best_params[0], gamma=best_params[1]),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name}: test accuracy = {acc:.4f}")
    print(confusion_matrix(y_test, y_pred))
    print()

print("Summary:")
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name}: {acc:.4f}")